# Anki French Frequency Sort

Reorder the new cards in an Anki deck by real-world French frequency.

Words are normalized to their dictionary form (lemma) by Claude (Batches API), scored against Lexique 3.83's lemma-level frequency corpus (with a wordfreq fallback), and the new cards are repositioned via AnkiConnect's local HTTP API.

**Design rule:** the LLM produces the *lemma*; the corpus produces the *number*. The LLM never invents frequency values.

Run the cells top to bottom.

**Prereqs:**
- `pip install -r requirements.txt` (anthropic, wordfreq, requests)
- Anki **open** with the AnkiConnect add-on (code 2055492159)
- `export ANTHROPIC_API_KEY=...`
- **Back up** your collection first (File → Export → .colpkg)

## [0] Imports

In [12]:
import os, re, csv, math, html, json, time, requests
import pandas as pd
from wordfreq import zipf_frequency   # fallback for words Lexique doesn't list
from anthropic import Anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

## [1] CONFIG — set these for your collection

In [ ]:
DECK_QUERY   = 'deck:"Français" is:new'   # which new cards to sort
SOURCE_FIELD = "Front"                  # field holding the French word/expression
MODEL        = "claude-haiku-4-5"       # cheap & plenty here — "claude-opus-4-8" for max accuracy
LEXIQUE_COL  = "freqlemfilms2"          # subtitles/everyday; "freqlemlivres" = books

DRY_RUN         = True     # True = preview only, writes NOTHING
AUTO_REPOSITION = False    # False = preview the planned order only (safe)
                           # True  = script sets new-card order directly (see caveats)

## [2] AnkiConnect bridge + helpers

In [4]:
def anki(action, **params):
    r = requests.post("http://localhost:8765",
                      json={"action": action, "version": 6, "params": params}, timeout=60).json()
    if r.get("error"):
        raise RuntimeError(f"AnkiConnect {action}: {r['error']}")
    return r["result"]

def clean(s):                       # Anki fields contain HTML
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", html.unescape(s)).strip()

## [1.5] Connectivity check (optional)

In [5]:
print("AnkiConnect version:", anki("version"))   # → 6 if the bridge is up
print(len(anki("findCards", query=DECK_QUERY)), "new cards match your query")

AnkiConnect version: 6
1056 new cards match your query


## [2.5] Lexique frequency index — lemma-level, aggregated across inflected forms

In [6]:
LEXIQUE_TSV = os.environ.get("LEXIQUE_TSV", "Lexique383.tsv")
LEXIQUE_URL = "http://www.lexique.org/databases/Lexique383/Lexique383.tsv"

if not os.path.exists(LEXIQUE_TSV):                       # one-time ~30 MB download
    print("Downloading Lexique383 …")
    resp = requests.get(LEXIQUE_URL, timeout=180); resp.raise_for_status()
    with open(LEXIQUE_TSV, "wb") as fh:
        fh.write(resp.content)

def _f(x):
    try: return float(x)
    except (TypeError, ValueError): return 0.0

lex_lemma, lex_ortho = {}, {}        # lowercased form -> best per-million frequency
with open(LEXIQUE_TSV, encoding="utf-8", newline="") as fh:
    for row in csv.DictReader(fh, delimiter="\t"):
        f = _f(row.get(LEXIQUE_COL))
        lem, ort = row["lemme"].strip().lower(), row["ortho"].strip().lower()
        if lem and f > lex_lemma.get(lem, -1.0): lex_lemma[lem] = f   # max over homographs
        if ort and f > lex_ortho.get(ort, -1.0): lex_ortho[ort] = f
print(f"Lexique loaded: {len(lex_lemma)} lemmas via {LEXIQUE_COL}.")

def _zipf_equiv(f):                  # per-million -> Zipf scale (log10 per-billion)
    return math.log10(f) + 3.0 if f > 0 else 0.0

def token_freq(tok):                 # one word: Lexique lemma -> Lexique surface -> wordfreq
    f = lex_lemma.get(tok, lex_ortho.get(tok))
    return _zipf_equiv(f) if f is not None else zipf_frequency(tok, "fr")

Lexique loaded: 46947 lemmas via freqlemfilms2.


## [3] Pull the new cards

In [7]:
items = [{"card_id": c["cardId"], "note_id": c["note"],
          "raw": clean(c["fields"][SOURCE_FIELD]["value"])}
         for c in anki("cardsInfo", cards=anki("findCards", query=DECK_QUERY))]
print(f"{len(items)} new cards found.")

1056 new cards found.


## [4] Normalize each distinct entry with Claude (Batches API)

In [8]:
SYSTEM = (
    "You are a French lexicographer normalizing vocabulary flashcard entries. "
    "For the given entry, return its dictionary citation form (lemma): nouns as masculine "
    "singular with NO article, verbs as the infinitive, adjectives as masculine singular; "
    "strip articles and elisions (le/la/les/un/une/des, l', d'); lowercase unless a proper noun. "
    "For a multi-word expression, give the canonical form of the whole expression. "
    "Flag is_suspect=true only if the entry looks like a typo or is not valid French, and put "
    "the corrected French in 'suggestion' (empty string otherwise)."
)
SCHEMA = {
    "type": "object",
    "properties": {
        "lemma":      {"type": "string"},
        "pos":        {"type": "string",
                       "enum": ["noun","verb","adjective","adverb","pronoun","preposition",
                                "conjunction","determiner","interjection","phrase","other"]},
        "is_suspect": {"type": "boolean"},
        "suggestion": {"type": "string"},
    },
    "required": ["lemma","pos","is_suspect","suggestion"],
    "additionalProperties": False,
}

raws = sorted({it["raw"] for it in items if it["raw"]})   # dedupe identical text
idx2raw = {f"w{i}": r for i, r in enumerate(raws)}
client = Anthropic()

batch = client.messages.batches.create(requests=[
    Request(custom_id=cid, params=MessageCreateParamsNonStreaming(
        model=MODEL, max_tokens=300, system=SYSTEM,
        messages=[{"role": "user", "content": raw}],
        output_config={"format": {"type": "json_schema", "schema": SCHEMA}},
    )) for cid, raw in idx2raw.items()
])
print("Batch:", batch.id, "— polling (usually minutes)…")

while client.messages.batches.retrieve(batch.id).processing_status != "ended":
    time.sleep(20)

norm = {}
for res in client.messages.batches.results(batch.id):
    if res.result.type == "succeeded":
        txt = next((b.text for b in res.result.message.content if b.type == "text"), "{}")
        norm[idx2raw[res.custom_id]] = json.loads(txt)
    else:
        print("FAILED:", idx2raw[res.custom_id], res.result.type)

Batch: msgbatch_01M6X8JsxKCQRj1qmFBigaoM — polling (usually minutes)…


## [5] Attach lemma + frequency (Lexique lemma-level; rarest component word for phrases)

In [9]:
def freq_of(lemma):
    toks = [t for t in lemma.lower().split() if t]
    return min((token_freq(t) for t in toks), default=0.0)

for it in items:
    n = norm.get(it["raw"], {})
    it["lemma"]   = n.get("lemma", "")
    it["suspect"] = n.get("is_suspect", False)
    it["suggest"] = n.get("suggestion", "")
    it["zipf"]    = freq_of(it["lemma"]) if it["lemma"] else 0.0

ordered = sorted(items, key=lambda x: x["zipf"], reverse=True)   # most frequent first

## [6] Preview (always) — stop here if DRY_RUN

In [14]:
preview = (pd.DataFrame(ordered, columns=["raw", "lemma", "zipf", "suspect", "suggest"])
             .rename(columns={"suspect": "typo?", "suggest": "suggestion"}))

# Flag entries the pipeline couldn't place, right in the table:
#   no lemma → Claude returned nothing / batch request failed / Front was blank
#   no freq  → got a lemma, but it's in neither Lexique nor wordfreq (zipf 0.0)
preview["status"] = "ok"
preview.loc[preview["zipf"] == 0, "status"] = "no freq"
preview.loc[preview["lemma"] == "", "status"] = "no lemma"   # more specific; wins over "no freq"

print(f"{len(preview)} cards, sorted most-frequent first · "
      f"{int(preview['typo?'].sum())} possible typos · "
      f"{int((preview['status'] == 'no lemma').sum())} no lemma · "
      f"{int((preview['status'] == 'no freq').sum())} no freq.")

with pd.option_context("display.max_rows", None):
    display(preview.round({"zipf": 2}))

assert not DRY_RUN, "DRY_RUN is True — nothing written. Review above, then set DRY_RUN=False."

1056 cards, sorted most-frequent first · 400 possible typos · 0 no lemma · 8 no freq.


,raw,lemma,zipf,typo?,suggestion,status
0,en plus (de),en plus,6.61,False,,ok
1,en plus (de),en plus,6.61,False,,ok
2,en plus,en plus,6.61,False,,ok
3,en plus,en plus,6.61,False,,ok
4,prendre de,prendre de,6.28,False,,ok
5,prendre de,prendre de,6.28,False,,ok
6,prendre à,prendre à,6.28,False,,ok
7,prendre à,prendre à,6.28,False,,ok
8,un ton.s,ton,6.24,True,ton,ok
9,un ton.s,ton,6.24,True,ton,ok


AssertionError: DRY_RUN is True — nothing written. Review above, then set DRY_RUN=False.

## [7] Reposition the new cards

In [ ]:
# Reposition the new cards so the most frequent come first.
if AUTO_REPOSITION:
    # Sets each new card's `due` (its position in the new queue) directly.
    # Requires the deck's new-card sort order to be position/"order added" based.
    for pos, it in enumerate(ordered, start=1):
        anki("setSpecificValueOfCard",
             card=it["card_id"], keys=["due"], newValues=[str(pos)],
             warning_check=True)   # acknowledges this is a low-level write
    print("Repositioned", len(ordered), "cards.")
else:
    print(f"AUTO_REPOSITION is False — nothing written. {len(ordered)} cards ready; "
          "set AUTO_REPOSITION=True to apply the new order.")